# 03 | SOH Target Design and Sensitivity Analysis

## Study objective

This notebook develops a transparent state-of-health definition from the available SOFC measurements and evaluates whether it remains stable across cells, operating regimes and experimental conditions.

The analysis compares alternative SOH formulations, reference choices and smoothing assumptions to identify a forecasting target that represents measured degradation without overstating what the experiment can reliably support.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns

from sofc_health.targets.rul import add_rul_target
from sofc_health.targets.soh import add_soh_targets

ROOT = Path.cwd()

if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

features = pd.read_parquet(ROOT / "data" / "processed" / "features.parquet")

modeling = add_soh_targets(features)

print("Feature-table shape:", features.shape)
print("Modeling-table shape:", modeling.shape)
print(
    "SOH columns:",
    [column for column in modeling.columns if column.startswith("soh_")],
)

## Math

For positive performance proxy $p_{c,k}$, $SOH_{c,k}=100p_{c,k}/p_{c,1}$. The composite is $SOH^{comp}_{c,k}=\exp(\frac{1}{M}\sum_m\log SOH^{(m)}_{c,k})$. Normalization makes trajectories comparable but assumes the first assessment is a meaningful baseline. The geometric mean penalizes disagreement more than an arithmetic mean.

In [ ]:
target_columns = [column for column in modeling.columns if column.startswith("soh_")]

long = modeling.melt(
    id_vars=["cell_id", "assessment_index"],
    value_vars=target_columns,
    var_name="target",
    value_name="soh_pct",
)

g = sns.relplot(
    data=long,
    x="assessment_index",
    y="soh_pct",
    hue="cell_id",
    col="target",
    col_wrap=2,
    kind="line",
    facet_kws={"sharey": False},
    height=4,
)

for ax in g.axes.flat:
    ax.axhline(
        80,
        color="black",
        linestyle="--",
        linewidth=1,
    )

In [ ]:
violations = modeling.groupby("cell_id")[target_columns].apply(
    lambda frame: (frame.diff() > 0).sum()
)

display(violations)

sensitivity = []

for threshold in (75.0, 80.0, 85.0):
    labeled = add_rul_target(
        modeling,
        threshold_pct=threshold,
    )

    sensitivity.append(
        {
            "threshold_pct": threshold,
            "observed_eol_cells": int(
                (~labeled.groupby("cell_id")["rul_right_censored"].first()).sum()
            ),
        }
    )

pd.DataFrame(sensitivity)

In [ ]:
violation_rates = {}

for cell_id, cell_frame in modeling.groupby("cell_id"):
    cell_frame = cell_frame.sort_values("assessment_index")
    cell_rates = {}

    for target in target_columns:
        differences = cell_frame[target].diff().dropna()

        cell_rates[target] = 100 * (differences > 0).mean() if len(differences) else float("nan")

    violation_rates[cell_id] = cell_rates

violation_rates = (
    pd.DataFrame.from_dict(violation_rates, orient="index").rename_axis("cell_id").round(1)
)

print("Local SOH increase rate (% of valid transitions)")
display(violation_rates)

labeled_85 = add_rul_target(
    modeling,
    threshold_pct=85.0,
)

observed_eol_85 = (
    labeled_85.loc[
        ~labeled_85["rul_right_censored"],
        ["cell_id", "eol_assessment"],
    ]
    .drop_duplicates()
    .sort_values("cell_id")
)

print("Cells with sustained EOL at the 85% threshold")
display(observed_eol_85)

## Conclusion and modeling decision

This notebook evaluated three performance-retention SOH candidates and one composite SOH target:

- Transient performance-current retention
- Maximum IV power retention
- IV current-density retention at 0.70 V
- Geometric-mean composite SOH

For cell \(c\), assessment \(k\), and performance indicator \(m\), SOH was defined relative to the first valid assessment:

$$
\mathrm{SOH}_{c,k}^{(m)}
=
100
\frac{p_{c,k}^{(m)}}{p_{c,1}^{(m)}}
$$

The composite target was calculated as:

$$
\mathrm{SOH}_{c,k}^{\mathrm{comp}}
=
\exp\left[
\frac{1}{M_{c,k}}
\sum_{m=1}^{M_{c,k}}
\ln\left(\mathrm{SOH}_{c,k}^{(m)}\right)
\right]
$$

The geometric mean prevents one strong modality from completely compensating for a weak modality. However, two of the three components are derived from the same IV experiment, so the composite contains partially redundant information. This limitation will be tested later through target and modality ablation.

### Main findings

1. The regular-redox cells generally show gradual performance decline, but most trajectories remain above 80% SOH during the observed experiment.

2. R1 and R2 behave differently from the regular cells. They show stronger short-term recovery, fluctuation, or protocol sensitivity.

3. Composite SOH increased during approximately 12.8% to 31.7% of valid transitions for regular cells, compared with 44.9% for R1 and 42.9% for R2.

4. These local SOH increases may represent measurement variability, conditioning, reversible recovery, operating-condition effects, or redox-related behaviour. Monotonic decline should therefore not be imposed before these effects are understood.

5. No cell showed a sustained composite-SOH crossing at the 75% or 80% thresholds.

6. R1 showed a sustained crossing at the 85% threshold beginning at assessment 3, but subsequently recovered above 100%. This is a mathematical threshold excursion, not defensible evidence of irreversible end of life.

7. The existence of an apparent EOL event is highly sensitive to the chosen threshold. The threshold must therefore be based on an engineering, warranty, safety, or operational requirement rather than selected to create more failure labels.

### Final decision

The composite SOH will be retained as the primary target for multi-horizon performance-retention forecasting:

$$
\widehat{\mathrm{SOH}}_{c,k+h}
=
f\left(\mathbf{x}_{c,1:k}\right)
$$

Here:

- \(c\) represents the physical cell.
- \(k\) represents the current degradation assessment.
- \(h\) represents the forecast horizon.
- \(\mathbf{x}_{c,1:k}\) represents all information available up to assessment \(k\).

Forecast horizons will be expressed in future assessments because reliable operating-hour intervals are unavailable.

Regular and randomized-redox cells will be reported separately. A pooled model will be accepted only if cross-regime validation demonstrates stable performance and no systematic regime-specific bias.

Exact RUL regression will not be treated as a validated task because no irreversible 75% or 80% EOL event was observed. The lifetime observations are right-censored:

$$
\mathrm{RUL}_{c,k}
>
k_{\mathrm{last},c}-k
$$

This inequality means that the cell survived beyond the final observed assessment, but its actual end-of-life assessment is unknown.

The defensible next task is leakage-safe, multi-horizon SOH forecasting, followed by uncertainty estimation and sensitivity analysis. Any RUL analysis will be presented only as a censored-survival methodology demonstration, not as validated lifetime prediction.